# NeuroGraphDB — NV-Embed-v2 인코딩 (Colab)

GPU가 필요한 작업은 **임베딩뿐**이다. 여기서 한 번 계산해 Hub에 올리면
이후 실험은 전부 CPU에서 무료로 돈다.

## 무엇을 만드는가

**① 후속 질의 임베딩** — 지금까지 2홉 검색을 BM25로 한 것은 후속 질의를
임베딩할 방법이 없어서였다(NV-Embed-v2는 7.85B라 로컬 불가).
같은 공간에 넣으면 dense로 2홉을 돌릴 수 있고, 그것이 2홉 품질을 직접 올린다.
→ `emb/{ds}_followups_r{1,2}_NV-Embed-v2.npy`  (6,000건, 수 분)

**② PopQA** — PropRAG이 보고하는 네 번째 데이터셋. 빠뜨리면
"유리한 것만 골랐다"는 말을 듣는다. 문단 8,676 + 질문 1,000.
→ `emb/popqa_NV-Embed-v2_{P,Q}.npy`

## 런타임

**L4 또는 A100을 고를 것.** T4(16GB)는 fp16 가중치 15.7GB만으로 꽉 차서 실패한다.
런타임 → 런타임 유형 변경 → L4 GPU.

예상 30분 내외, **2~3 컴퓨팅 단위**.

## 사전 준비

왼쪽 🔑 **보안 비밀**에 `HF_TOKEN`을 추가하고 노트북 접근을 켤 것.
**쓰기 권한이 있는 토큰**이어야 한다(결과를 Hub에 올린다).

In [ ]:
# NV-Embed-v2의 remote code는 구버전 transformers API를 쓴다.
# 최신 버전에서는 _tied_weights_keys 문제로 로드가 실패하므로 핀을 고정한다.
# datasets 4.x는 torchcodec을 끌고 와 충돌하므로 <4로 묶는다.
!pip install -q "transformers==4.42.4" "datasets>=2.14,<4" "huggingface_hub>=0.28" einops
import torch, subprocess
print(torch.__version__, torch.cuda.is_available())
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
import json, time, numpy as np, torch
from pathlib import Path
from google.colab import userdata
from huggingface_hub import login, hf_hub_download, HfApi

login(userdata.get("HF_TOKEN"))
api = HfApi()

RESULTS = "goethe0101/neurographdb-results"
OFFICIAL = "osunlp/HippoRAG_v2"
TAG = "NV-Embed-v2"
# 질의 instruction. **job_e1.py와 한 글자도 달라선 안 된다** —
# 다르면 새 벡터가 기존 캐시와 같은 공간에 있지 않게 된다.
INSTR = "Instruct: Given a question, retrieve documents that best answer the question\nQuery: "

def exists(path):
    try:
        hf_hub_download(RESULTS, path, repo_type="dataset"); return True
    except Exception:
        return False

def push(arr, path):
    f = f"/content/{path.replace('/','_')}"
    np.save(f, arr.astype(np.float32))
    api.upload_file(path_or_fileobj=f, path_in_repo=path,
                    repo_id=RESULTS, repo_type="dataset")
    print(f"  올림: {path}  {arr.shape}")

In [ ]:
# 모델 적재 (~15GB 내려받는다. 5분쯤 걸린다)
from transformers import AutoModel
t0 = time.time()
mdl = AutoModel.from_pretrained("nvidia/NV-Embed-v2", trust_remote_code=True,
                                torch_dtype=torch.float16).to("cuda").eval()
print(f"적재 {time.time()-t0:.0f}s")

def encode(items, instruction="", batch=8):
    """job_e1.py의 encode()와 동일해야 한다 — max_length 512, L2 정규화, fp16."""
    out, t = [], time.time()
    for a in range(0, len(items), batch):
        with torch.no_grad():
            e = mdl.encode(items[a:a+batch], instruction=instruction, max_length=512)
            e = torch.nn.functional.normalize(e, p=2, dim=1)
        out.append(e.float().cpu().numpy())
        if (a // batch) % 100 == 0:
            print(f"    {a}/{len(items)}  {time.time()-t:.0f}s", flush=True)
    return np.concatenate(out).astype(np.float32)

In [ ]:
# ── ① 후속 질의 임베딩 ───────────────────────────────────────────────
# 후속 질의는 **질의**이므로 원래 질문과 같은 instruction을 붙인다.
# 그래야 문단 벡터 P와 같은 방식으로 비교된다.
# NONE / 빈 응답은 검색을 돌리지 않으므로 영벡터로 두고 평가 쪽에서 건너뛴다.

for ds in ("musique", "2wiki", "hotpotqa"):
    for rnd in (1, 2):
        out_path = f"emb/{ds}_followups_r{rnd}_{TAG}.npy"
        if exists(out_path):
            print(f"건너뜀(이미 있음): {out_path}"); continue
        src = (f"decomp/{ds}_followups_llama31.json" if rnd == 1
               else f"decomp/{ds}_followups_llama31_r{rnd}.json")
        try:
            fu = json.load(open(hf_hub_download(RESULTS, src, repo_type="dataset")))["followups"]
        except Exception as e:
            print(f"건너뜀(원본 없음): {src}"); continue
        keep = [i for i, f in enumerate(fu)
                if f and not f.strip().upper().startswith("NONE")]
        print(f"{ds} R{rnd}: {len(fu)}건 중 인코딩 대상 {len(keep)}")
        E = np.zeros((len(fu), 4096), dtype=np.float32)
        if keep:
            E[keep] = encode([fu[i] for i in keep], instruction=INSTR)
        push(E, out_path)

In [ ]:
# ── ② PopQA ─────────────────────────────────────────────────────────
# PropRAG이 보고하는 네 번째 데이터셋(Recall@5 55.3). 단일홉이라 우리 방법이
# 힘을 못 쓰는 자리지만, 빼면 유리한 것만 골랐다는 말을 듣는다.

if exists(f"emb/popqa_{TAG}_P.npy") and exists(f"emb/popqa_{TAG}_Q.npy"):
    print("건너뜀: PopQA 임베딩이 이미 있다")
else:
    corpus = json.load(open(hf_hub_download(OFFICIAL, "popqa_corpus.json", repo_type="dataset")))
    qs = json.load(open(hf_hub_download(OFFICIAL, "popqa.json", repo_type="dataset")))
    print(f"PopQA: 문단 {len(corpus):,}  질문 {len(qs):,}")
    # 문단 형식도 job_e1.py와 동일해야 한다: "제목. 본문", instruction 없음
    P = encode([f"{c['title']}. {c['text']}" for c in corpus])
    Q = encode([r["question"] for r in qs], instruction=INSTR)
    push(P, f"emb/popqa_{TAG}_P.npy")
    push(Q, f"emb/popqa_{TAG}_Q.npy")

In [ ]:
# ── 검증 — 기존 캐시와 **같은 공간**에 있는지 확인한다 ────────────────
# 설정이 어긋났으면 여기서 드러난다. MuSiQue 질의를 다시 인코딩해
# 기존 캐시와 코사인이 1에 가까운지 본다.
Qold = np.load(hf_hub_download(RESULTS, f"emb/musique_{TAG}_Q.npy", repo_type="dataset"))
qs = json.load(open(hf_hub_download(OFFICIAL, "musique.json", repo_type="dataset")))
Qnew = encode([r["question"] for r in qs[:32]], instruction=INSTR)
cos = (Qnew * Qold[:32]).sum(1)
print(f"기존 캐시와의 코사인: 최소 {cos.min():.4f}  평균 {cos.mean():.4f}")
print("0.999 이상이어야 한다. 아니면 설정이 어긋난 것이니 알려줄 것.")